# Title: (PalestineIsraelWar-Cleaning)

## Introduction
Dataset about all events in Palestine and Israel War 

## Process

### Import libraries

In [ ]:
%pip install ruff

In [ ]:
%pip install ydata-profiling

In [ ]:
import pandas as pd  # noqa: F401
import numpy as np  # noqa: F401
import matplotlib.pyplot as plt  # noqa: F401
import seaborn as sns  # noqa: F401
from pandas_profiling import ProfileReport  # noqa: F401

### Load Data

In [ ]:
df = pd.read_csv(
    "../../1_datasets/data/01_category_war_events_data/gaza_war_events/palestine_israel_conflict/data.csv"
)
df.head()

In [ ]:
df.columns  # You can refer to the data dictionary for more information about each column!

### Explore the dates (what period of time does it cover and how much we need?)

In [ ]:
print("Oldest Event", df["event_date"].min())
print("Most recent Event", df["event_date"].max())

So basically we don't need that all events before 07/10/2023, we will drop anything before that

In [ ]:
df["event_date"] = pd.to_datetime(df["event_date"])
df = df.loc[df["event_date"] >= "2023-10-07"]

In [ ]:
print("Oldest Event", df["event_date"].min())
print("Most recent Event", df["event_date"].max())

We are also focusing now on the damage on Gaza, so we will drop everything any other data

In [ ]:
df = df.loc[df["admin1"] == "Gaza Strip"]

### Viewing and modifing column names

In [ ]:
pd.DataFrame(
    df.columns
)  # You can refer to the data dictionary for more information about each column!

As we see, no all the columns' names are descriptive

In [ ]:
df = df.rename(
    columns={
        "event_id_cnty": "event_id",
        "event_date": "date",
        "time_precision": "date_precision",
        "event_type": "event_group",
        "sub_event_type": "event_subtype",
        "actor1": "primary_targeting_actor",
        "assoc_actor_1": "assoc_targeting_actor",
        "inter1": "primary_targeting_actor_type",
        "actor2": "primary_targeted_actor",
        "assoc_actor_2": "assoc_targeted_actor",
        "inter2": "primary_targeted_actor_type",
        "interaction": "interaction_type",
        "civilian_targeting": "civilian_targeted",
        "iso": "country_iso",
        "region": "region",
        "country": "country",
        "admin1": "administrative_division_1",
        "admin2": "administrative_division_2",
        "admin3": "administrative_division_3",
        "source": "sources",
        "source_scale": "source_scope",
        "fatalities": "fatality_count",
        "tags": "tags",
        "timestamp": "data_timestamp",
    }
)

### Explore Data for missing values

In [ ]:
df.info()

In [ ]:
pd.DataFrame(df.isnull().sum()).loc[df.isnull().sum() != 0]

### We can use the pandas profiling for this columns

In [ ]:
profile = ProfileReport(df, title="Data Quality Profiling Report")
profile

In [ ]:
profile.to_file("data_quality_report.html")

َQuick action: The region column has constant value: "Middle East" so it doesn't have any contribution to the value of our dataset, we'll drop it

In [ ]:
df.drop("region", axis=1, inplace=True)

civilian_targeted has constant value "Civilian targeting"
We will look into it on data exploration phase

### Visualize the null values

In [ ]:
null_columns = [
    "assoc_targeting_actor",
    "primary_targeted_actor",
    "assoc_targeted_actor",
    "primary_targeted_actor_type",
    "civilian_targeted",
    "administrative_division_1",
    "administrative_division_2",
    "administrative_division_3",
    "tags",
]
null_prects = df[null_columns].isnull().sum() / df.shape[0] * 100

# Plot
plt.figure(figsize=(10, 6))
null_prects.sort_values(ascending=False).plot(
    kind="bar", color="tomato", edgecolor="black"
)
plt.title("Null Values precentage per Column")
plt.ylabel("Number of Missing Values")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.show()

### Handling Missing Values

1. We will drop columns with high null values precentage
2. We will replace na with Unkown value in other columns

In [ ]:
df.drop(columns=["administrative_division_3", "tags"], inplace=True)

As quick processing step We will replace the missing values with "Unkown" in the columns because we didn't explore the data enough

In [ ]:
df.fillna("Unknown", inplace=True)

In [ ]:
df.isnull().sum()

### Handling Duplicates

In [ ]:
df.duplicated().any()

### Handling Outliers

In [ ]:
# outliers in this column are actually real numbers
plt.figure(figsize=(6, 10))
df.boxplot(column="fatality_count")
plt.title("Boxplot of Fatalities")
plt.ylabel("Number of Fatalities")
plt.grid(True)
plt.show()

### Convert Data Types

In [ ]:
df.dtypes

We notice that date_precision, geo_percision is in numeric type, but it is cateogrical so we will map the values into textual values

In [ ]:
df["geo_precision_label"] = df["geo_precision"].map(
    {1: "Exact location", 2: "Admin division", 3: "Broad/Unknown area"}
)
df["date_precision_label"] = df["date_precision"].map(
    {1: "Exact date", 2: "Approximate date", 3: "Broad/Unknown date"}
)

In [ ]:
df.drop(columns=["geo_precision", "date_precision"], inplace=True)

In [ ]:
df.rename(
    columns={
        "geo_precision_label": "geo_precision",
        "date_precision_label": "date_precision",
    },
    inplace=True,
)

### Save Cleaned Data

In [ ]:
df.to_csv(
    "../../1_datasets/data/clean_datasets/palestine_israel_war_cleaned_data.csv",
    index=False,
)